In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/FYP_SR_Data")

for folder in ["DIV2K", "Set5", "Set14", "BSD100", "Urban100"]:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)

print(DATA_ROOT)

/content/drive/MyDrive/FYP_SR_Data


In [ ]:
!pip -q install -U kagglehub

In [5]:
from pathlib import Path
import kagglehub

div2k_dir = Path("/content/drive/MyDrive/FYP_SR_Data/DIV2K")
div2k_dir.mkdir(parents=True, exist_ok=True)

path = kagglehub.dataset_download(
    "takihasan/div2k-dataset-for-super-resolution",
    output_dir=str(div2k_dir)
)

print("Dataset saved at:", path)

100%|██████████| 4.94G/4.94G [01:15<00:00, 70.4MB/s]

Extracting files...


Dataset saved at: /content/drive/MyDrive/FYP_SR_Data/DIV2K


In [8]:
!pip -q install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 11.3 MB/s eta 0:00:00


In [9]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")
datasets = ["Set5", "Set14", "BSD100", "Urban100"]

for name in datasets:
    destination = data_root / name
    snapshot_download(
        repo_id=f"eugenesiow/{name}",
        repo_type="dataset",
        local_dir=str(destination),
        allow_patterns=["data/*"],
    )
    print(f"{name} downloaded")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Set5 downloaded


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

BSD100 downloaded


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Urban100 downloaded


In [10]:
import tarfile

for name in datasets:
    dataset_dir = data_root / name
    for archive in (dataset_dir / "data").glob("*.tar.gz"):
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(dataset_dir)
    print(f"{name} extracted")

/tmp/ipykernel_5566/2070269884.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dataset_dir)


Set5 extracted
Set14 extracted
BSD100 extracted
Urban100 extracted


In [11]:
from pathlib import Path
from time import perf_counter

from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")
result_dir = data_root / "results" / "baseline"
result_dir.mkdir(parents=True, exist_ok=True)

hr_path = data_root / "Set5" / "Set5_HR" / "baby.png"
lr_path = data_root / "Set5" / "Set5_LR_x2" / "baby.png"

hr_image = Image.open(hr_path).convert("RGB")
lr_image = Image.open(lr_path).convert("RGB")

start = perf_counter()

bicubic_image = lr_image.resize(
    hr_image.size,
    Image.Resampling.BICUBIC
)

runtime_ms = (perf_counter() - start) * 1000

bicubic_image.save(result_dir / "Set5_baby_bicubic_x2.png")

hr_array = __import__("numpy").array(hr_image)
sr_array = __import__("numpy").array(bicubic_image)

psnr = peak_signal_noise_ratio(hr_array, sr_array, data_range=255)
ssim = structural_similarity(
    hr_array,
    sr_array,
    channel_axis=2,
    data_range=255
)

print(f"PSNR: {psnr:.4f} dB")
print(f"SSIM: {ssim:.4f}")
print(f"Runtime: {runtime_ms:.2f} ms")
print(f"Saved to: {result_dir}")

PSNR: 35.6163 dB
SSIM: 0.9500
Runtime: 13.48 ms
Saved to: /content/drive/MyDrive/FYP_SR_Data/results/baseline


In [12]:
import numpy as np
import pandas as pd

from pathlib import Path
from time import perf_counter
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")

hr_dir = data_root / "Set5" / "Set5_HR"
lr_dir = data_root / "Set5" / "Set5_LR_x2"

output_dir = data_root / "results" / "baseline" / "Set5_x2"
output_dir.mkdir(parents=True, exist_ok=True)

In [13]:
def evaluate_bicubic(image_name):
    hr_image = Image.open(hr_dir / image_name).convert("RGB")
    lr_image = Image.open(lr_dir / image_name).convert("RGB")

    start = perf_counter()

    sr_image = lr_image.resize(
        hr_image.size,
        Image.Resampling.BICUBIC
    )

    runtime_ms = (perf_counter() - start) * 1000

    hr_array = np.array(hr_image)
    sr_array = np.array(sr_image)

    psnr = peak_signal_noise_ratio(
        hr_array,
        sr_array,
        data_range=255
    )

    ssim = structural_similarity(
        hr_array,
        sr_array,
        channel_axis=2,
        data_range=255
    )

    sr_image.save(output_dir / f"{Path(image_name).stem}_bicubic_x2.png")

    return {
        "image": image_name,
        "scale": "x2",
        "method": "bicubic",
        "psnr": psnr,
        "ssim": ssim,
        "runtime_ms": runtime_ms,
    }

In [14]:
image_names = sorted(
    image.name for image in hr_dir.glob("*.png")
)

records = [
    evaluate_bicubic(image_name)
    for image_name in image_names
]

results_df = pd.DataFrame(records)
results_df

,image,scale,method,psnr,ssim,runtime_ms
0,baby.png,x2,bicubic,35.616256,0.950003,9.090744
1,bird.png,x2,bicubic,34.871968,0.967712,4.715152
2,butterfly.png,x2,bicubic,26.116475,0.902398,4.500261
3,head.png,x2,bicubic,31.509056,0.820040,3.421674
4,woman.png,x2,bicubic,30.820043,0.948868,3.603125


In [15]:
average_results = results_df[[
    "psnr",
    "ssim",
    "runtime_ms"
]].mean()

print(f"Average PSNR: {average_results['psnr']:.4f} dB")
print(f"Average SSIM: {average_results['ssim']:.4f}")
print(f"Average Runtime: {average_results['runtime_ms']:.2f} ms")

Average PSNR: 31.7868 dB
Average SSIM: 0.9178
Average Runtime: 5.07 ms


In [16]:
results_df.to_csv(
    data_root / "results" / "baseline" / "Set5_x2_bicubic.csv",
    index=False
)

In [17]:
import numpy as np
import pandas as pd

from pathlib import Path
from time import perf_counter
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")

scale = "x3"

hr_dir = data_root / "Set5" / "Set5_HR"
lr_dir = data_root / "Set5" / "Set5_LR_x3"

output_dir = data_root / "results" / "baseline" / "Set5_x3"
output_dir.mkdir(parents=True, exist_ok=True)

In [18]:
def evaluate_bicubic(image_name):
    hr_image = Image.open(hr_dir / image_name).convert("RGB")
    lr_image = Image.open(lr_dir / image_name).convert("RGB")

    start = perf_counter()

    sr_image = lr_image.resize(
        hr_image.size,
        Image.Resampling.BICUBIC
    )

    runtime_ms = (perf_counter() - start) * 1000

    hr_array = np.array(hr_image)
    sr_array = np.array(sr_image)

    psnr = peak_signal_noise_ratio(
        hr_array,
        sr_array,
        data_range=255
    )

    ssim = structural_similarity(
        hr_array,
        sr_array,
        channel_axis=2,
        data_range=255
    )

    sr_image.save(output_dir / f"{Path(image_name).stem}_bicubic_x3.png")

    return {
        "image": image_name,
        "scale": "x3",
        "method": "bicubic",
        "psnr": psnr,
        "ssim": ssim,
        "runtime_ms": runtime_ms,
    }

In [19]:
image_names = sorted(
    image.name for image in hr_dir.glob("*.png")
)

records = [
    evaluate_bicubic(image_name)
    for image_name in image_names
]

results_df = pd.DataFrame(records)
results_df

,image,scale,method,psnr,ssim,runtime_ms
0,baby.png,x3,bicubic,32.494283,0.898310,8.256624
1,bird.png,x3,bicubic,30.609280,0.919648,2.626370
2,butterfly.png,x3,bicubic,22.732223,0.804567,1.797092
3,head.png,x3,bicubic,29.998209,0.753016,2.213688
4,woman.png,x3,bicubic,27.199325,0.889164,2.127687


In [20]:
average_results = results_df[[
    "psnr",
    "ssim",
    "runtime_ms"
]].mean()

print(f"Average PSNR: {average_results['psnr']:.4f} dB")
print(f"Average SSIM: {average_results['ssim']:.4f}")
print(f"Average Runtime: {average_results['runtime_ms']:.2f} ms")

Average PSNR: 28.6067 dB
Average SSIM: 0.8529
Average Runtime: 3.40 ms


In [21]:
results_df.to_csv(
    data_root / "results" / "baseline" / "Set5_x3_bicubic.csv",
    index=False
)

In [22]:
import numpy as np
import pandas as pd

from pathlib import Path
from time import perf_counter
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")

scale = "x4"

hr_dir = data_root / "Set5" / "Set5_HR"
lr_dir = data_root / "Set5" / "Set5_LR_x4"

output_dir = data_root / "results" / "baseline" / "Set5_x4"
output_dir.mkdir(parents=True, exist_ok=True)

In [23]:
def evaluate_bicubic(image_name):
    hr_image = Image.open(hr_dir / image_name).convert("RGB")
    lr_image = Image.open(lr_dir / image_name).convert("RGB")

    start = perf_counter()

    sr_image = lr_image.resize(
        hr_image.size,
        Image.Resampling.BICUBIC
    )

    runtime_ms = (perf_counter() - start) * 1000

    hr_array = np.array(hr_image)
    sr_array = np.array(sr_image)

    psnr = peak_signal_noise_ratio(
        hr_array,
        sr_array,
        data_range=255
    )

    ssim = structural_similarity(
        hr_array,
        sr_array,
        channel_axis=2,
        data_range=255
    )

    sr_image.save(output_dir / f"{Path(image_name).stem}_bicubic_x4.png")

    return {
        "image": image_name,
        "scale": "x4",
        "method": "bicubic",
        "psnr": psnr,
        "ssim": ssim,
        "runtime_ms": runtime_ms,
    }

In [24]:
image_names = sorted(
    image.name for image in hr_dir.glob("*.png")
)

records = [
    evaluate_bicubic(image_name)
    for image_name in image_names
]

results_df = pd.DataFrame(records)
results_df

,image,scale,method,psnr,ssim,runtime_ms
0,baby.png,x4,bicubic,30.416118,0.842698,19.015875
1,bird.png,x4,bicubic,28.068146,0.859798,2.299019
2,butterfly.png,x4,bicubic,20.900521,0.718410,1.590291
3,head.png,x4,bicubic,28.963417,0.701769,1.891146
4,woman.png,x4,bicubic,25.102982,0.826870,2.028579


In [25]:
average_results = results_df[[
    "psnr",
    "ssim",
    "runtime_ms"
]].mean()

print(f"Average PSNR: {average_results['psnr']:.4f} dB")
print(f"Average SSIM: {average_results['ssim']:.4f}")
print(f"Average Runtime: {average_results['runtime_ms']:.2f} ms")

Average PSNR: 26.6902 dB
Average SSIM: 0.7899
Average Runtime: 5.36 ms


In [26]:
results_df.to_csv(
    data_root / "results" / "baseline" / "Set5_x4_bicubic.csv",
    index=False
)